In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch

from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode
import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal","Denmark"]
#days = ["day1", "day2", "day3", "day4", "day5"]


days = ["day1"]


PRED_LEN = 96
MAX_CONTEXT = 10000

# ============================================================
# LOAD DAY CUTOFFS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# INIT TabPFN-TS PIPELINE (LOCAL)
# ============================================================
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.LOCAL)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

rmse_results = []

for country in countries:
    print("Processing country:", country)
    country_start_time = time.time()

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # --------------------------------------------------------
    # COUNTRY-SPECIFIC FEATURES
    # --------------------------------------------------------
    country_features = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "direct_radiation",
    ]

    if country == "Denmark":
        country_features.append("price_eur_kwh")

    households = [c for c in df.columns if c not in country_features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            hist_idx = df.index[df.index < cutoff]
            s_train = df.loc[hist_idx, household].astype(float)

            if s_train.dropna().shape[0] < 10:
                continue

            context_df = pd.DataFrame({
                "item_id": household,
                "timestamp": s_train.index,
                "target": s_train.values,
            }).tail(MAX_CONTEXT).reset_index(drop=True)

            pred_df = pipeline.predict_df(
                context_df=context_df,
                prediction_length=PRED_LEN,
            )

            pred_df_reset = pred_df.reset_index()

            ts_pred = pd.to_datetime(pred_df_reset["timestamp"])

            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=ts_pred)

            y_pred = pred_df_reset[0.5].to_numpy()
            predictions_df_all_households[household] = y_pred

            y_true = df.loc[ts_pred, household].to_numpy()

            if np.isnan(y_true).any():
                continue

            rmse_households.append(root_mean_squared_error(y_true, y_pred))

        if predictions_df_all_households is None or len(rmse_households) == 0:
            print(f"      No predictions produced for {country} {day}. Skipping.")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output = rf"{OUT_DIR}\TabPFNTS_UNIV_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

    # ========================================================
    # COUNTRY RUNTIME
    # ========================================================
    country_runtime = time.time() - country_start_time

    print(
        f"\nTotal runtime for {country}: "
        f"{country_runtime:.2f} seconds"
    )

    # ========================================================
    # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
    # ========================================================
    json_path = os.path.join(
        OUT_DIR,
        f"time_spend_{country}.json"
    )

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            runtime_dict = json.load(f)
    else:
        runtime_dict = {}

    if "Foundational" not in runtime_dict:
        runtime_dict["Foundational"] = {}

    runtime_dict["Foundational"]["TabPFN"] = country_runtime

    with open(json_path, "w") as f:
        json.dump(runtime_dict, f, indent=4)

    print(f"Saved/updated runtime JSON: {json_path}")
